In [ ]:
import pandas as pd
import json
from azure.storage.blob import BlobServiceClient

# =========================
# CONFIG
# =========================
STORAGE_CONNECTION_STRING = "ENTER_CONNECTION_STRING"
CONTAINER_NAME = "CONTAINER_NAME"
OUTPUT_CSV = "telemetry_dataset.csv"
OUTPUT_BLOB_NAME = "telemetry_dataset.csv"   # name of CSV inside storage

# =========================
# CONNECT TO STORAGE
# =========================
blob_service_client = BlobServiceClient.from_connection_string(STORAGE_CONNECTION_STRING)
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

rows = []

# =========================
# READ ALL JSON / JSONL FILES
# =========================
for blob in container_client.list_blobs():
    if blob.name.endswith(".json") or blob.name.endswith(".jsonl"):
        print(f"Reading: {blob.name}")

        blob_client = container_client.get_blob_client(blob.name)
        content = blob_client.download_blob().readall().decode("utf-8", errors="ignore")

        for line in content.splitlines():
            line = line.strip()
            if not line:
                continue

            try:
                obj = json.loads(line)

                body = obj.get("Body", {})
                metrics = body.get("Metrics", {})
                system_props = obj.get("SystemProperties", {})

                row = {
                    "EnqueuedTimeUtc": obj.get("EnqueuedTimeUtc"),
                    "SystemDeviceId": system_props.get("connectionDeviceId"),
                    "Timestamp": body.get("Timestamp"),
                    "DeviceId": body.get("DeviceId"),
                    "DeviceType": body.get("DeviceType"),
                    "Label": body.get("Label"),
                    "AttackId": body.get("AttackId"),

                    "AveragePacketRate": metrics.get("AveragePacketRate"),
                    "TotalFailedLogins": metrics.get("TotalFailedLogins"),
                    "SuccessfulLogins": metrics.get("SuccessfulLogins"),
                    "FailedLoginRate": metrics.get("FailedLoginRate"),
                    "UniqueSourceIps": metrics.get("UniqueSourceIps"),
                    "FailedToSuccessRatio": metrics.get("FailedToSuccessRatio"),
                    "UniquePortsAccessed": metrics.get("UniquePortsAccessed"),
                    "ConnectionAttemptsPerSecond": metrics.get("ConnectionAttemptsPerSecond"),
                    "AverageConnectionDurationMs": metrics.get("AverageConnectionDurationMs"),
                    "NewConnectionsPerSecond": metrics.get("NewConnectionsPerSecond"),
                    "TrafficVolumeBytes": metrics.get("TrafficVolumeBytes"),
                    "OutgoingBytes": metrics.get("OutgoingBytes"),
                    "IncomingBytes": metrics.get("IncomingBytes"),
                    "OutgoingIncomingRatio": metrics.get("OutgoingIncomingRatio"),
                    "AverageCpuUsage": metrics.get("AverageCpuUsage"),
                    "TimeOfDay": metrics.get("TimeOfDay"),
                    "AfterHoursActivity": metrics.get("AfterHoursActivity"),
                }

                rows.append(row)

            except Exception as e:
                print("Skipping invalid JSON line:", e)

# =========================
# CREATE DATAFRAME
# =========================
df = pd.DataFrame(rows)

# =========================
# CLEAN NUMERIC COLUMNS
# =========================
numeric_cols = [
    "DeviceType",
    "Label",
    "AveragePacketRate",
    "TotalFailedLogins",
    "SuccessfulLogins",
    "FailedLoginRate",
    "UniqueSourceIps",
    "FailedToSuccessRatio",
    "UniquePortsAccessed",
    "ConnectionAttemptsPerSecond",
    "AverageConnectionDurationMs",
    "NewConnectionsPerSecond",
    "TrafficVolumeBytes",
    "OutgoingBytes",
    "IncomingBytes",
    "OutgoingIncomingRatio",
    "AverageCpuUsage",
    "TimeOfDay",
    "AfterHoursActivity"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["DeviceId", "DeviceType", "Label"])

# =========================
# SAVE CSV LOCALLY
# =========================
df.to_csv(OUTPUT_CSV, index=False)

print(f"\nDone. Extracted {len(df)} rows.")
print(f"Local CSV saved as: {OUTPUT_CSV}")

# =========================
# UPLOAD CSV TO STORAGE ACCOUNT
# =========================
csv_blob_client = container_client.get_blob_client(OUTPUT_BLOB_NAME)

with open(OUTPUT_CSV, "rb") as data:
    csv_blob_client.upload_blob(data, overwrite=True)

print(f"CSV uploaded to storage as: {OUTPUT_BLOB_NAME}")

# Optional preview
df.head()

Reading: anthony-hub-project/01/2026/04/13/09/20.json
Reading: anthony-hub-project/01/2026/04/13/09/22.json
Reading: anthony-hub-project/01/2026/04/13/10/07.json
Reading: anthony-hub-project/01/2026/04/13/10/09.json

Done. Extracted 354 rows.
Local CSV saved as: telemetry_dataset.csv
CSV uploaded to storage as: telemetry_dataset.csv


,EnqueuedTimeUtc,SystemDeviceId,Timestamp,DeviceId,DeviceType,Label,AttackId,AveragePacketRate,TotalFailedLogins,SuccessfulLogins,...,ConnectionAttemptsPerSecond,AverageConnectionDurationMs,NewConnectionsPerSecond,TrafficVolumeBytes,OutgoingBytes,IncomingBytes,OutgoingIncomingRatio,AverageCpuUsage,TimeOfDay,AfterHoursActivity
0,2026-04-13T09:20:08.9850000Z,gateway-01,2026-04-13T09:20:08.451076Z,WS-001,0,0,None,50.913305,0,7,...,2.295706,728.631108,0.907480,77171.425490,49156.426802,28014.998688,1.754647,27.660312,19,1
1,2026-04-13T09:20:09.2660000Z,gateway-01,2026-04-13T09:20:10.0381418Z,WS-002,0,0,None,51.808395,0,6,...,4.543269,372.756669,0.000000,64888.388843,11370.129958,53518.258885,0.212453,16.164012,19,1
2,2026-04-13T09:20:09.5000000Z,gateway-01,2026-04-13T09:20:10.2469625Z,WS-003,0,0,None,85.871465,0,1,...,1.143118,950.415428,0.000000,103231.920054,52947.443959,50284.476096,1.052958,32.333051,19,1
3,2026-04-13T09:20:09.7350000Z,gateway-01,2026-04-13T09:20:10.4913643Z,WS-004,0,1,attack_c56be,131.671167,3,4,...,0.083028,332.908797,1.800803,78282.302705,18361.306745,59920.995961,0.306425,27.908558,19,1
4,2026-04-13T09:20:09.9530000Z,gateway-01,2026-04-13T09:20:10.7120988Z,WS-005,0,0,None,75.320507,1,4,...,4.240950,1397.666447,2.702480,143505.419351,42433.916456,101071.502895,0.419841,16.916678,19,1
